[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Naviden/Introduction-to-XAI/blob/main/Practice/Core_XAI_Exercises.ipynb)

[Colab setup: dependencies and repository files](https://github.com/Naviden/Introduction-to-XAI/blob/main/docs/REPRODUCIBILITY.md#google-colab)

**Validated reading copy.** Run the [source notebook](../../../Practice/Core_XAI_Exercises.ipynb) in its original directory. This snapshot includes checked outputs; source SHA-256: `8e1f548ee782fd8363f73c70a346b260919ffccc71910d3e7a2d5bb9640b28f7`. Timings describe this recorded environment, not a performance guarantee.

# Core XAI exercises

Complete three exercises on additive-model units, robustness/masking and counterfactual validity. Install `requirements-tabular.txt`; the setup runs offline on CPU with generated data. **The TODO cells are intentionally unfinished.** Successful execution of the template only verifies setup, not a completed assignment.

Prerequisites: train/test splits, probabilities versus logits, and the [intrinsic-model comparison](../../../Tools/InterpretableModels_examples/Interpretable_Models_Comparison.ipynb). Allow roughly one teaching session; the computational examples are small.

Submit the completed notebook and brief answers. Instructor material is separate in `Practice/Instructor/`; consult it after attempting the exercises.

## Shared setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import SplineTransformer

SEED = 42
rng = np.random.default_rng(SEED)
signal = rng.normal(size=600)
X = pd.DataFrame({'signal': signal, 'proxy': signal + rng.normal(scale=0.08, size=600),
                  'context': rng.normal(size=600)})
latent_logit = 1.5 * X.signal - 0.5 * X.proxy + np.sin(2 * X.context) + 0.3
y = pd.Series(rng.binomial(1, expit(latent_logit)), name='class_1')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
assert set(X_train.index).isdisjoint(X_test.index)
features = X.columns.tolist()
additive = Pipeline([
    ('basis', ColumnTransformer([(f'term_{j}', SplineTransformer(n_knots=4, degree=3, knots='quantile', include_bias=False), [name])
                                 for j, name in enumerate(features)])),
    ('classifier', LogisticRegression(C=0.7, max_iter=2000, random_state=SEED)),
])
additive.fit(X_train, y_train)
forest = RandomForestClassifier(n_estimators=80, max_depth=4, min_samples_leaf=8, n_jobs=2, random_state=SEED)
forest.fit(X_train, y_train)
query = X_test.iloc[[0]]
print('Synthetic data only; class 1 is the explained output.')
display(pd.Series({'additive_test_AUC': roc_auc_score(y_test, additive.predict_proba(X_test)[:,1]),
                   'forest_test_AUC': roc_auc_score(y_test, forest.predict_proba(X_test)[:,1])}))

Synthetic data only; class 1 is the explained output.


additive_test_AUC    0.797783
forest_test_AUC      0.794150
dtype: float64

## Exercise 1 — additive model units and exact reconstruction (30 points)

The fitted spline classifier has one term per original feature and a logistic link. Recover each term's contribution from `additive.named_steps['basis'].output_indices_` and the fitted logistic coefficients. Center terms over **training rows only**, adding their means to the intercept.

1. Reconstruct the selected case's logit from the centered intercept and three terms; verify it against `decision_function`.
2. Apply the sigmoid and verify the class-1 probability. Plot the three centered contributions with correctly labeled units.
3. Explain why summing contributions in log-odds is not the same as summing changes in probability. Is `sigmoid(mean(logit))` generally the same as `mean(probability)`?
4. Transfer the reasoning to binary EBM: what do its intercept and evaluated term scores sum to? If an EBM contains a pairwise interaction term, may every displayed term still be described as a function of one input?

**Deliverable:** code checks, one contribution plot, and a short interpretation that separates exact model decomposition from causal explanation. Do not assume that this spline implementation is the EBM algorithm.

In [2]:
# TODO: compute and center the per-feature terms using training rows.
# TODO: verify the selected logit and probability, then plot the contributions.

**Your interpretation:** replace this placeholder.

## Exercise 2 — randomness, correlated features and masking (35 points)

Keep the fitted forest and test set fixed.

1. Run held-out ROC-AUC permutation importance using five different shuffle seeds. Report means and variation, stating exactly what the variation measures.
2. Compare shuffling `signal` alone with shuffling the pair (`signal`, `proxy`) together using the **same row permutation**. Why does the latter preserve their mutual relationship?
3. Define an illustrative support check using the training 99th percentile of `abs(signal − proxy)`. Measure the fraction exceeding it in the original test set, independently shuffled data, and jointly shuffled data. This is a diagnostic for the designed relationship, not a general out-of-distribution detector.
4. Explain why a small permutation score for one correlated feature does not establish irrelevance, and why neither shuffle experiment identifies a causal effect.

**Deliverable:** a summary table/plot, a support-check table and a statement distinguishing shuffle variability from a confidence interval over future populations.

In [3]:
# TODO: run repeated permutation importance without refitting the forest.
# TODO: compare independent and joint shuffling and evaluate the support diagnostic.

**Your interpretation:** replace this placeholder.

## Exercise 3 — counterfactual validity is more than a prediction flip (35 points)

Use the fictional classifier and candidate pool below. The target is class 1 under a probability threshold of 0.5. `prior_score` records history and is immutable. `study_hours` may only increase; `commute_hours` may only decrease. All values must stay within the declared bounds.

1. Check prediction validity, immutability, bounds and allowed directions **separately** for every candidate. A candidate is admissible only if all checks pass.
2. Among admissible candidates, rank normalized L1 distance from the original, dividing each absolute change by its allowed range width. State that this finds the closest **provided** candidate under a chosen metric, not a globally minimal recourse.
3. Report pairwise normalized L1 distance between the admissible candidates as a simple diversity description. Explain why diversity alone is insufficient.
4. Describe one assumption required to turn these hypothetical changes into a real action plan. Explain why satisfying this toy model's constraints does not prove real-world feasibility or causal benefit.

In [4]:
# A deliberately fictional course-completion classifier; not an intervention model.
def completion_probability(frame):
    return expit(-3 + 0.7 * frame['study_hours'] + 0.2 * frame['prior_score'] - 0.6 * frame['commute_hours'])

original = pd.Series({'study_hours': 2., 'prior_score': 2., 'commute_hours': 2.})
candidates = pd.DataFrame([
    [6., 2., 1.], [7., 2., 2.], [2., 20., 2.],
    [30., 2., 2.], [2., 2., 1.], [10., 2., 3.],
], columns=original.index, index=['candidate_A', 'candidate_B', 'candidate_C', 'candidate_D', 'candidate_E', 'candidate_F'])
lower_bounds = pd.Series({'study_hours': 0., 'prior_score': 0., 'commute_hours': 0.})
upper_bounds = pd.Series({'study_hours': 12., 'prior_score': 10., 'commute_hours': 4.})
print('Original probability:', float(completion_probability(original.to_frame().T).iloc[0]))
display(candidates)

Original probability: 0.08317269649392235


,study_hours,prior_score,commute_hours
candidate_A,6.0,2.0,1.0
candidate_B,7.0,2.0,2.0
candidate_C,2.0,20.0,2.0
candidate_D,30.0,2.0,2.0
candidate_E,2.0,2.0,1.0
candidate_F,10.0,2.0,3.0


In [5]:
# TODO: build one column for each validity/constraint check.
# TODO: filter admissible candidates before comparing distance and diversity.

**Your interpretation:** replace this placeholder.

## Self-check before submission

- Every table names the model output, class and units.
- Preprocessing, centering and support thresholds use only training data.
- Randomness is controlled and repeated where the exercise asks for it.
- Numerical reconstruction, predictive validity and action constraints are different checks.
- Conclusions stay within the evidence; a failed explanation diagnostic is reported rather than hidden.